## Part 1: Foundations with Python using Fitbit Data

**USF INFORMS Student Chapter**  
Sponsored by the INFORMS Quality, Statistics, and Reliability (QSR) Section  
Friday, October 9, 2026 · ENG 3

Fitness apps tell us to reach **10,000 steps** a day and to get **7+ hours of sleep** a night. Many people assume the two goals support each other: a more active day should lead to a better night.

**Our question:** Do more active days come with better sleep?

We will answer it with real wearable data, following the same workflow data scientists use every day.

| Step | What we will do |
|---|---|
| 1 | Understand what the data contain |
| 2 | Load the tables, check them, and combine them |
| 3 | Explore distributions, relationships, and outliers |
| 4 | State a hypothesis and test it in several ways |
| 5 | Compare each person with themselves |

**Keep this distinction in mind:** Finding an association between activity and sleep is a statistical task. Claiming that one causes the other also depends on how the data were collected and what else could explain the pattern.

**How to use this notebook:** run cells from top to bottom with **Shift + Enter**. **Discuss** and **Predict** prompts are questions to think about before moving on; click **Reveal the answer** to check your thinking.

## Setup

We use four standard libraries, all preinstalled in Colab: **pandas** (tables), **numpy** (math), **matplotlib** (plots), and **scipy.stats** (statistical tests).

Running this notebook **outside Colab**, for example in Jupyter on your laptop? Remove the `#` in the next cell and run it once.

In [ ]:
# Only needed outside Colab: remove the "#" and run once
# !pip install pandas numpy matplotlib scipy

In [ ]:
# Import the tools we will use for tables, math, plots, and statistical tests.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["figure.figsize"] = (8, 4)
GREEN, GOLD = "#006747", "#CFC493"

BASE_URL = "https://raw.githubusercontent.com/cvnh23/bootcamp-2026/main/"

## 1. About the data

About 30 Fitbit users shared their tracker data through a survey distributed on Amazon Mechanical Turk. We use one month of it: **April 12 to May 12, 2016**.

Source: Furberg, Brinton, Keating & Ortiz (2016), *Crowd-sourced Fitbit datasets*, Zenodo, [doi:10.5281/zenodo.53894](https://doi.org/10.5281/zenodo.53894).

The data come in several tables at different levels of detail:

| Level | What it represents | Table we use |
|---|---|---|
| Person | One participant, identified by `Id` | All tables |
| Day of activity | One person on one day: steps, distance, active minutes, calories | `dailyActivity_merged.csv` |
| Day of sleep | One person's sleep recorded for one date | `sleepDay_merged.csv` |
| Heart-rate reading | One reading every few seconds | `heartrate_seconds_merged.csv.gz` (optional) |

**Columns we will use most:**

| Column | Table | Meaning |
|---|---|---|
| `Id` | Both | Participant identifier |
| `ActivityDate`, `SleepDay` | Activity, Sleep | The date of the record, stored as text |
| `TotalSteps` | Activity | Steps counted that day |
| `TotalDistance` | Activity | Estimated distance covered |
| `VeryActiveMinutes` … `SedentaryMinutes` | Activity | Minutes spent at each activity intensity |
| `Calories` | Activity | Estimated calories burned |
| `TotalSleepRecords` | Sleep | Number of separate sleep periods recorded that day, such as a night plus a nap |
| `TotalMinutesAsleep` | Sleep | Minutes classified as asleep |
| `TotalTimeInBed` | Sleep | Minutes from going to bed to getting up, including time awake |

**What these numbers are:** a wrist-worn tracker does not measure sleep or steps directly. It records motion (and heart rate on some models), and Fitbit's algorithms turn those signals into estimates. A day with very few steps might be a rest day, or a day the tracker sat on a charger.

**Discuss:** each person contributes up to about 31 days. Is each row an independent observation?

<details>
<summary>Reveal the answer</summary>

No. Days from the same person are related: someone who usually walks a lot or sleeps little will do so on many days. Most of the tests in Section 4 treat every day as independent, so we will revisit this in Section 5.

</details>

**Discuss:** before loading anything, list two reasons a recorded value could differ from what actually happened.

<details>
<summary>Reveal the answer</summary>

The tracker was not worn or ran out of battery; it was worn for only part of the day or night; the algorithm misread the motion (for example, lying still while awake counted as sleep, or arm movement counted as steps); or the user logged an activity by hand. These are measurement questions, and we will meet some of them when we look at outliers.

</details>

## 2. Load and combine the tables

We read two tables directly from the bootcamp's GitHub repository.

In [ ]:
# Read the daily activity and sleep tables, then show their sizes.
daily = pd.read_csv(BASE_URL + "dailyActivity_merged.csv")
sleep = pd.read_csv(BASE_URL + "sleepDay_merged.csv")

print("daily:", daily.shape, "  sleep:", sleep.shape)   # (rows, columns)
daily.head()

**Discuss:** look at the first rows. What does one row of `daily` represent? Which columns identify the person and the day?

<details>
<summary>Reveal the answer</summary>

One row is **one person on one day**. `Id` identifies the person and `ActivityDate` identifies the day. Together they form the row's identity, which is exactly what we will use to match activity with sleep.

</details>

### Check data types and duplicates

`.info()` lists each column's type and the number of non-missing values. Dates often load as plain **text**, which Python cannot sort or compare correctly, so we convert them.

In [ ]:
# List each column's name, data type, and number of non-missing values.
daily.info()

**Predict before running:** `ActivityDate` looks like `4/12/2016`, while `SleepDay` looks like `4/12/2016 12:00:00 AM`. If we matched the two tables on these text columns, how many rows would match?

<details>
<summary>Reveal the answer</summary>

None. As text, `4/12/2016` and `4/12/2016 12:00:00 AM` are different strings, so a merge on them would find no matches. Converting both to real dates gives a common `Date` column that matches correctly.

</details>

In [ ]:
# Convert text dates to real dates, count people and dates, and remove duplicate sleep rows.
daily["Date"] = pd.to_datetime(daily["ActivityDate"], format="%m/%d/%Y")
sleep["Date"] = pd.to_datetime(sleep["SleepDay"], format="%m/%d/%Y %I:%M:%S %p")

print("People:", daily["Id"].nunique(), "| Dates:", daily["Date"].min().date(), "to", daily["Date"].max().date())
print("Duplicate sleep rows:", sleep.duplicated().sum())
sleep = sleep.drop_duplicates()

**Discuss:** why remove duplicate sleep rows **before** merging?

<details>
<summary>Reveal the answer</summary>

A duplicated night would appear twice in the merged table, counting the same sleep twice and making the sample look larger than it is. Duplicates usually come from export or logging glitches, not from two real nights.

</details>

### Combine the tables

To compare activity and sleep, we need them in **one table**. We **merge** on the two columns that identify a row: person (`Id`) and `Date`. We also convert sleep and time in bed to hours, which are easier to read than minutes.

**Predict before running:** the activity table has about 940 rows and the sleep table about 410. Will the merged table be closer to 940 or to 410?

In [ ]:
# Keep only person-days found in both tables, then express sleep and time in bed in hours.
df = daily.merge(sleep, on=["Id", "Date"], how="inner")
df["SleepHours"] = df["TotalMinutesAsleep"] / 60
df["HoursInBed"] = df["TotalTimeInBed"] / 60

print(f"{len(df)} days with both activity and sleep, from {df['Id'].nunique()} people")
df[["Id", "Date", "TotalSteps", "SleepHours", "HoursInBed"]].head().round(2)

**Discuss:**

1. We started with 940 activity days but have only about 410 rows now. Where did the rest go?
2. `how="inner"` keeps only matching rows. Could the people who log sleep be **different** from those who do not?

<details>
<summary>Reveal the answer</summary>

1. An inner merge keeps only days that appear in **both** tables. Many people never logged sleep (only 24 of 33 did), so their days drop out.
2. Yes. People who wear their tracker to bed may be more health-conscious, for example. Our conclusions apply to **sleep trackers**, not everyone. This is called **selection bias**, and it is worth stating whenever you merge or filter.

</details>

### Optional: what large data look like

The same study also recorded heart rate every few seconds. That file has about **2.5 million rows**. We will not use it for our question, but run the next cell to see how pandas handles it.

In [ ]:
# Load the large second-by-second heart-rate file (optional).
hr = pd.read_csv(BASE_URL + "heartrate_seconds_merged.csv.gz")   # ~10 seconds
print(f"{len(hr):,} heart-rate readings from {hr['Id'].nunique()} people, "
      f"{hr.memory_usage(deep=True).sum()/1e6:.0f} MB in memory")

**Discuss:** if we wanted to add heart rate to our sleep question, what would we need to do with these 2.5 million readings first?

<details>
<summary>Reveal the answer</summary>

Summarize them to the same level as our table: **one row per person per day**, for example average or resting heart rate. Turning many readings into a few summary numbers is called feature engineering.

</details>

## 3. Explore the data

Before testing anything, **look at the data**:

1. What do the values look like? (**distributions**)
2. How do the variables relate? (**visualization**)
3. Is anything strange or suspicious? (**outliers**)

### Check distributions

`.describe()` gives the classic summary: count, mean, standard deviation, minimum, quartiles, and maximum.

In [ ]:
# Summarize steps, sleep, and time in bed.
df[["TotalSteps", "SleepHours", "HoursInBed"]].describe().round(1)

**Discuss:**

1. Compare the mean and the median (the `50%` row) for each variable. What does a gap between them suggest?
2. Look at the minimum of `SleepHours`. Is it a plausible night of sleep?

<details>
<summary>Reveal the answer</summary>

1. When the mean is pulled away from the median, a tail of unusual values is likely on that side. A mean below the median points to a long left tail; a mean above it points to a long right tail.
2. A minimum of around an hour is not a typical night. Keep it in mind: it may be a nap or a partial record, and we examine it in the outlier section.

</details>

In [ ]:
# Plot the distribution of daily steps and of nightly sleep, with the two goals marked.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(df["TotalSteps"], bins=30, color=GREEN, edgecolor="white")
axes[0].axvline(10000, color="black", ls="--", label="10,000 steps")
axes[0].set(title="Daily steps", xlabel="Steps", ylabel="Days"); axes[0].legend()

axes[1].hist(df["SleepHours"], bins=30, color=GOLD, edgecolor="white")
axes[1].axvline(7, color="black", ls="--", label="7 hours")
axes[1].set(title="Sleep per night", xlabel="Hours asleep"); axes[1].legend()
plt.tight_layout(); plt.show()

**Discuss:** using the dashed lines, roughly what share of days reach 10,000 steps? What share reach 7 hours of sleep?

<details>
<summary>Reveal the answer</summary>

- **Steps:** about **40%** of days reach 10,000 steps (165 of 410). Most days fall short of the goal.
- **Sleep:** about **56%** of nights reach 7 hours (229 of 410). Just over half meet the goal.

So the "active" group (165 days) will be smaller than the "less active" group (245 days). Unequal group sizes are fine for the tests we use, but they're worth knowing before we compare. The counts printed under the boxplot below confirm these numbers.

</details>

Many statistical tests assume the data are **normally distributed** (bell-shaped). Two ways to check:

- **Q-Q plot:** if the data are normal, the points fall on the straight line.
- **Shapiro–Wilk test:** a small p-value (< 0.05) is evidence the data are **not** normal.

In [ ]:
# Check normality two ways: Q-Q plots and the Shapiro-Wilk test.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ["TotalSteps", "SleepHours"]):
    stats.probplot(df[col], dist="norm", plot=ax)
    ax.set_title(f"Q-Q plot: {col}")
    ax.get_lines()[0].set(color=GREEN, markersize=3); ax.get_lines()[1].set(color="black")
plt.tight_layout(); plt.show()

for col in ["TotalSteps", "SleepHours"]:
    print(f"Shapiro-Wilk {col}: p = {stats.shapiro(df[col]).pvalue:.1e}")

**Discuss:**

1. Which variable departs more from a bell shape? Where do the Q-Q points bend away from the line?
2. Both Shapiro–Wilk p-values are tiny. Does that mean we cannot use any test that assumes normality?

<details>
<summary>Reveal the answer</summary>

1. Steps are fairly symmetric here, with only mild bending at the ends of the Q-Q plot. Sleep is bell-shaped in the middle but has a **long left tail**: a cluster of very short "nights" (1–3 hours) that pulls the Q-Q points far below the line on the left.
2. Not necessarily. With hundreds of data points, Shapiro–Wilk flags even small departures from normality. Also, the t-test compares **means**, and means of large samples are approximately normal anyway (the **Central Limit Theorem**). The safe habit: also run a test that does **not** assume normality and see whether the conclusion changes. We do exactly that in Section 4.

</details>

### Visualize the relationship

Now the question itself: **do steps and sleep move together?** A scatter plot shows every day as one dot.

**Predict before running:** if active days come with better sleep, which direction should the cloud of points slope?

In [ ]:
# Plot every person-day as one dot, with the two goals marked.
plt.scatter(df["TotalSteps"], df["SleepHours"], color=GREEN, alpha=0.5)
plt.axvline(10000, color="gray", ls="--"); plt.axhline(7, color="gray", ls="--")
plt.xlabel("Daily steps"); plt.ylabel("Hours asleep"); plt.title("Each dot is one person-day")
plt.show()

**Discuss:** is there a clear pattern? Why can a trend be hard to judge by eye with hundreds of points?

<details>
<summary>Reveal the answer</summary>

Any trend is hard to see: the points form a wide cloud. With many overlapping points, a weak trend is easy to miss or to imagine. Group summaries and statistical tests give a more reliable answer than our eyes, which is why we compare groups next and test them in Section 4.

</details>

Split days into two groups, **active** (10,000+ steps) and **less active**, and compare their sleep with a **boxplot**. The box holds the middle 50% of days, the line inside is the **median**, and dots beyond the whiskers are potential **outliers**.

In [ ]:
# Label each day as active or less active, then compare sleep between the groups.
df["Active"] = df["TotalSteps"] >= 10000

plt.boxplot([df.loc[~df["Active"], "SleepHours"], df.loc[df["Active"], "SleepHours"]],
            patch_artist=True, boxprops=dict(facecolor=GOLD), medianprops=dict(color=GREEN, lw=2))
plt.xticks([1, 2], ["< 10,000 steps", "10,000+ steps"])
plt.ylabel("Hours asleep"); plt.title("Sleep on less active vs. active days"); plt.show()

print(df.groupby("Active")["SleepHours"].agg(["count", "mean", "median"]).round(2))

**Discuss:**

1. Based on the plots, do active days seem to come with **more** or **less** sleep?
2. What do you notice at the **bottom** of both boxes?

<details>
<summary>Reveal the answer</summary>

1. Surprisingly, **less**: active days have a lower median sleep. This is the opposite of what many people expect, so we need to test it carefully.
2. Both groups have a cluster of dots far below the box, at around 1–3 hours. That is our next topic.

</details>

**Try if time permits:** change `10000` to `8000` in the cell above and rerun it. Does the direction of the difference change? The 10,000-step cut-off is a popular goal, not a scientific threshold.

### Outliers: error or real?

A common rule flags values beyond **1.5 × IQR** (interquartile range) from the box edges. An outlier rule only tells you **where to look**. You decide what to do by understanding **why** the value is unusual.

In [ ]:
# Flag nights beyond 1.5 x IQR from the quartiles, then list the shortest ones.
q1, q3 = df["SleepHours"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["SleepHours"] < low) | (df["SleepHours"] > high)]

print(f"IQR rule: flag sleep below {low:.1f} h or above {high:.1f} h  ->  {len(outliers)} flagged nights")
outliers[["Id", "Date", "SleepHours", "HoursInBed", "TotalSleepRecords"]].sort_values("SleepHours").round(2).head(8)

**Discuss:**

1. Look at `HoursInBed` for the shortest nights. What do you think happened on those nights?
2. Should we **delete** these rows, **keep** them, or **fix** them? What would you write in a report to justify the choice?

<details>
<summary>Reveal the answer</summary>

1. Time in bed is only about 1 hour, so these are most likely **naps**, a tracker that ran out of battery, or a band taken off during the night. They are not real full nights of sleep.
2. Since our question is about **nightly** sleep, it is reasonable to exclude records shorter than 3 hours, **and say so**. The very long nights (11+ hours) are plausible (weekends, illness), so we keep them. Rule of thumb: remove values that are **measurement problems**, keep values that are **real but unusual**, and always document both.

</details>

In [ ]:
# Keep nights of at least 3 hours; shorter records are likely naps or partial records.
clean = df[df["SleepHours"] >= 3].copy()
print(f"Kept {len(clean)} of {len(df)} nights ({len(df) - len(clean)} likely naps or partial records removed)")

**Takeaway:** the 3-hour cut-off is a judgment call. Report it, and check whether conclusions depend on it.

**Try if time permits:** after finishing Section 4, change `3` to `2` or `4` in the cell above and rerun the tests. If the conclusion holds, it does not hinge on this one choice. This check is called a **sensitivity analysis**.

## 4. Test the hypothesis

### State the hypothesis before running tests

- **H₀ (null):** sleep is the **same** on active (10,000+ steps) and less active days.
- **H₁ (alternative):** sleep **differs** between them.

We use a significance level of **α = 0.05**. A **p-value** is the probability of seeing a difference at least this large **if H₀ were true**. A small p-value means the data would be surprising under H₀.

Different tests answer slightly different versions of the question, so we run several and compare.

**Discuss:** why write the hypothesis down **before** looking at test results?

<details>
<summary>Reveal the answer</summary>

If we choose the hypothesis, the groups, or the test after seeing the results, we can keep trying options until something looks significant. Stating the plan first keeps the p-value meaningful. Running several tests is fine when we report all of them, as we do in the summary table.

</details>

### Welch's t-test: do the means differ?

Welch's t-test compares two group averages without assuming the groups have equal spread.

In [ ]:
# Compare mean sleep between active and less active days.
active = clean.loc[clean["Active"], "SleepHours"]
less   = clean.loc[~clean["Active"], "SleepHours"]

t = stats.ttest_ind(active, less, equal_var=False)
print(f"Mean sleep: active days {active.mean():.2f} h vs. less active days {less.mean():.2f} h")
print(f"Difference: {active.mean() - less.mean():+.2f} h  |  t = {t.statistic:.2f}, p = {t.pvalue:.1e}")

**Discuss:** the p-value tells us whether the difference is surprising under H₀. Does it tell us whether the difference **matters** in daily life?

<details>
<summary>Reveal the answer</summary>

No. Statistical significance is about evidence, not size. Look at the difference itself, in hours, and ask whether that much sleep would matter to a person. With large samples, even small differences can produce tiny p-values.

</details>

### Mann–Whitney U test: no normality needed

The Mann–Whitney U test works on **ranks** instead of raw values, so it does not assume a bell curve and is less sensitive to outliers.

**Predict before running:** will this test reach the same conclusion as the t-test?

In [ ]:
# Compare the two groups using ranks instead of raw hours.
u = stats.mannwhitneyu(active, less)
print(f"Median sleep: active {active.median():.2f} h vs. less active {less.median():.2f} h  |  p = {u.pvalue:.1e}")

**Discuss:** if the t-test and the Mann–Whitney test agree, what does that tell us about the normality concern from Section 3?

<details>
<summary>Reveal the answer</summary>

The conclusion does not depend on assuming normal data. When a test with the assumption and a test without it agree, we can be more confident the result is not an artifact of that assumption.

</details>

### Correlation: a trend, not just two groups

Splitting at 10,000 steps is arbitrary. **Spearman's ρ** measures whether more steps go with more (or less) sleep across the **whole range**, using ranks.

In [ ]:
# Measure whether more steps go with more or less sleep across the whole range.
rho = stats.spearmanr(clean["TotalSteps"], clean["SleepHours"])
print(f"Spearman rho = {rho.statistic:.2f}, p = {rho.pvalue:.1e}")

**Discuss:** why use Spearman's ρ rather than the more familiar Pearson correlation?

<details>
<summary>Reveal the answer</summary>

Spearman's ρ uses ranks, so it detects any steady upward or downward trend, not only a straight-line one, and a few extreme days have less influence. Pearson's r measures linear association on the raw values and is more sensitive to outliers and skew.

</details>

### Chi-square test: do the goals go together?

Turn both variables into yes/no: *reached 10,000 steps?* and *slept 7+ hours?* Then check whether the two are independent.

In [ ]:
# Cross-tabulate the two yes/no goals and test whether they are independent.
clean["Slept7"] = clean["SleepHours"] >= 7
table = pd.crosstab(clean["Active"], clean["Slept7"])
table.index, table.columns = ["< 10k steps", "10k+ steps"], ["< 7 h sleep", "7+ h sleep"]
print(table, "\n")
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"Chi-square = {chi2:.1f}, p = {p:.3f}")
print("Share of days with 7+ h sleep:", (table["7+ h sleep"] / table.sum(axis=1)).round(2).to_dict())

**Discuss:** what information do we lose when we turn steps and sleep into yes/no categories?

<details>
<summary>Reveal the answer</summary>

We lose the size of each value. A night of 6.9 hours and one of 2 hours both become "below 7 hours", while 6.9 and 7.0 land in different groups. Categories match the way goals are stated, but they usually give a test less power than the original numbers.

</details>

**Discuss:** four tests, all with small p-values. Does that mean we have **proven** that walking more makes you sleep less?

<details>
<summary>Reveal the answer</summary>

No, for two reasons:

- **Correlation is not causation.** A busy day (travel, deadlines, events) could cause *both* more walking *and* less sleep.
- **Our tests assumed every day is independent**, but each person contributes many days. If some people simply walk a lot *and* sleep little in general, the pooled tests mix up differences **between people** with changes **within a person**. We fix that next.

</details>

**Follow-up:** the four tests used the same days. Do four small p-values give four times the evidence?

<details>
<summary>Reveal the answer</summary>

No. They are four views of the same data, so they share the same weaknesses, including the independence problem. Agreement shows the result is robust to the choice of test, not that we have four separate confirmations.

</details>

## 5. Compare each person with themselves

For each person, compute their average sleep on active days and on less active days. Then use a **paired test**, which looks at each person's own difference. The paired t-test compares the average difference; the Wilcoxon signed-rank test is its rank-based version.

**Predict before running:** will the evidence be stronger or weaker than in the pooled tests?

In [ ]:
# Average each person's sleep on each type of day, then compare within each person.
per_person = clean.groupby(["Id", "Active"])["SleepHours"].mean().unstack().dropna()
per_person.columns = ["Less active days", "Active days"]
diff = per_person["Active days"] - per_person["Less active days"]

print(f"{len(per_person)} people had both kinds of days")
print(f"Average within-person difference: {diff.mean():+.2f} h  ({(diff < 0).sum()} of {len(diff)} slept less on active days)")
print(f"Paired t-test:        p = {stats.ttest_rel(per_person['Active days'], per_person['Less active days']).pvalue:.3f}")
print(f"Wilcoxon signed-rank: p = {stats.wilcoxon(per_person['Active days'], per_person['Less active days']).pvalue:.3f}")

for _, row in per_person.iterrows():
    plt.plot([0, 1], row.values, color=GREEN, alpha=0.6, marker="o")
plt.xticks([0, 1], per_person.columns); plt.xlim(-0.3, 1.3)
plt.ylabel("Average hours asleep"); plt.title("Each line is one person"); plt.show()

**Discuss:** in the plot, each line is one person. What does a line sloping downward mean? How many people show that pattern?

<details>
<summary>Reveal the answer</summary>

A downward line means that person slept less, on average, on their active days. The printed count shows how many of the people follow that direction. Check whether most lines agree, or whether a few steep lines dominate the picture.

</details>

### How much data do we really have?

We started with what looked like a large dataset. Here is how many observations actually informed each step:

| Stage | Observations | Independent people |
|---|---|---|
| Heart-rate file (optional) | 2,483,658 readings | 14 |
| Activity table | 940 person-days | 33 |
| After merging with sleep | 410 person-nights | 24 |
| After removing naps (< 3 h) | 388 person-nights | 21 |
| Paired comparison | 14 within-person differences | **14** |

The data are also **unbalanced**: people contribute anywhere from 1 to 31 nights, and the 5 most-tracked people account for about 38% of all nights.

**Discuss:** we have 388 nights. Why is the amount of independent evidence closer to 14–21 than to 388?

<details>
<summary>Reveal the answer</summary>

Nights from the same person are related: someone who sleeps little tends to do so on many nights. Repeated measurements of one person add less new information than measurements of new people. The pooled tests in Section 4 treat all 388 nights as independent, so their p-values are **too optimistic**. The paired test uses one number per person, which is why it is the most trustworthy here, and why our conclusion is "suggestive," not "proven." A **mixed-effects model** can use every night while accounting for repeated measures; it is a natural next step beyond this session.

</details>

### Put it all together

In [ ]:
#@title Summary table of all tests (run me)
summary = pd.DataFrame([
    ["Welch t-test",       "Do mean sleep hours differ?",                 "All days pooled",  t.pvalue],
    ["Mann-Whitney U",     "Do distributions differ (rank-based)?",       "All days pooled",  u.pvalue],
    ["Spearman rho",       "Do more steps go with less/more sleep?",      "All days pooled",  rho.pvalue],
    ["Chi-square",         "Are the 10k-step and 7-hour goals linked?",   "All days pooled",  p],
    ["Paired t-test",      "Does the same person sleep differently?",     f"{len(per_person)} people", stats.ttest_rel(per_person['Active days'], per_person['Less active days']).pvalue],
    ["Wilcoxon signed-rank","Same, rank-based",                           f"{len(per_person)} people", stats.wilcoxon(per_person['Active days'], per_person['Less active days']).pvalue],
], columns=["Test", "Question it answers", "Unit of analysis", "p-value"])
summary["Significant at 0.05?"] = np.where(summary["p-value"] < 0.05, "Yes", "No")
summary["p-value"] = summary["p-value"].map(lambda v: f"{v:.1e}" if v < 0.001 else f"{v:.3f}")
summary

**Discuss:**

1. Why is the evidence much **weaker** in the paired tests than in the pooled ones?
2. In one or two sentences, how would you answer our original question: *do more active days come with better sleep?*
3. Look back at the merge: does a sleep record dated April 20 describe the night **before** or **after** that day's steps? Why does it matter?

<details>
<summary>Reveal the answer</summary>

1. The paired tests use **one difference per person** (about 14 numbers instead of about 390 days), so they have less **statistical power**. They also remove differences **between** people, which were driving part of the pooled result. The direction is the same (most people sleep a bit less on active days), but the evidence is borderline: the two paired tests even land on opposite sides of 0.05.
2. Something like: *"In this sample, days with 10,000+ steps came with roughly 45–50 minutes **less** sleep overall, and about half that within the same person. This is an association, not proof of cause, and it comes from a small, self-selected group."*
3. The data do not tell us for sure. Depending on how Fitbit dates a sleep record, the "same date" may pair a day's steps with the **previous** night's sleep. If so, the question we actually tested is "do people walk more after a short night?", which is a different story. **Knowing how the data were recorded is part of the analysis.**

</details>

### What we gained

- **Understood the data:** what one row means, how the tables differ in level, and that tracker values are estimates.
- **Loaded and checked:** converted types, removed duplicates, and noticed what a merge drops.
- **Explored before testing:** distributions, plots, and outliers shaped which tests were appropriate.
- **Handled outliers with a reason:** removed measurement problems, kept real extremes, and documented both.
- **Matched the test to the question:** means, ranks, trends, categories, and paired designs.
- **Looked beyond the p-value:** independence, confounding, and how the data were recorded.

These are the building blocks of **Quality, Statistics, and Reliability**: trusting measurements, modeling variation, and drawing careful conclusions from data.

## References

- Furberg, R., Brinton, J., Keating, M., & Ortiz, A. (2016). *Crowd-sourced Fitbit datasets 03.12.2016–05.12.2016*. Zenodo. https://doi.org/10.5281/zenodo.53894
- pandas: [merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) and [to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html).
- SciPy: [Shapiro–Wilk](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html), [Welch's t-test](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html), [Mann–Whitney U](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mannwhitneyu.html), [Spearman's ρ](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html), [chi-square](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html), [paired t-test](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html), and [Wilcoxon signed-rank](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wilcoxon.html).

---
## Keep in touch
- **Join INFORMS QSR:** https://connect.informs.org/qsr/join-qsr3
- **QSR home** (webinars, tutorials, student competitions): https://connect.informs.org/qsr/home
- **USF INFORMS Student Chapter:** Instagram @informs_sc_usf · LinkedIn: informs-sc-usf